# Indian Object Translator — Google Colab

This notebook walks through YOLO object detection followed by translation into India's 22 Scheduled Languages.


In [ ]:
!pip install -q ultralytics gradio python-dotenv requests pillow pandas


In [ ]:
from ultralytics import YOLO
from PIL import Image
from google.colab import files

model = YOLO('yolo26n.pt')
print('YOLO loaded')


In [ ]:
uploaded = files.upload()
image_path = next(iter(uploaded))
image = Image.open(image_path).convert('RGB')
display(image)


In [ ]:
results = model(image, verbose=False)
result = results[0]

detections = []
for box in result.boxes:
    confidence = float(box.conf[0])
    if confidence < 0.40:
        continue
    class_id = int(box.cls[0])
    object_name = result.names[class_id]
    detections.append({'object': object_name, 'confidence': confidence})

detections


## Configure Google Translation

In Colab, open **Secrets** and create `GOOGLE_TRANSLATE_API_KEY`. Do not paste your key directly into a notebook cell.


In [ ]:
from google.colab import userdata

GOOGLE_TRANSLATE_API_KEY = userdata.get('GOOGLE_TRANSLATE_API_KEY')
print('API key available:', bool(GOOGLE_TRANSLATE_API_KEY))


In [ ]:
import requests

def translate_text(text, target_language):
    url = 'https://translation.googleapis.com/language/translate/v2'
    response = requests.post(
        url,
        params={'key': GOOGLE_TRANSLATE_API_KEY},
        json={
            'q': text,
            'source': 'en',
            'target': target_language,
            'format': 'text',
        },
        timeout=20,
    )
    response.raise_for_status()
    return response.json()['data']['translations'][0]['translatedText']

print('cat -> Hindi:', translate_text('cat', 'hi'))


In [ ]:
SCHEDULED_LANGUAGES = [
    ('Assamese', 'as'), ('Bengali', 'bn'), ('Bodo', 'brx'), ('Dogri', 'doi'),
    ('Gujarati', 'gu'), ('Hindi', 'hi'), ('Kannada', 'kn'), ('Kashmiri', 'ks'),
    ('Konkani', 'gom'), ('Maithili', 'mai'), ('Malayalam', 'ml'), ('Manipuri', 'mni-Mtei'),
    ('Marathi', 'mr'), ('Nepali', 'ne'), ('Odia', 'or'), ('Punjabi', 'pa'),
    ('Sanskrit', 'sa'), ('Santali', 'sat'), ('Sindhi', 'sd'), ('Tamil', 'ta'),
    ('Telugu', 'te'), ('Urdu', 'ur'),
]

# Google currently does not list Kashmiri and Santali in its NMT support.
GOOGLE_UNSUPPORTED = {'Kashmiri', 'Santali'}

object_name = detections[0]['object'] if detections else 'cat'
rows = []

for language_name, language_code in SCHEDULED_LANGUAGES:
    if language_name in GOOGLE_UNSUPPORTED:
        rows.append((language_name, '--', 'google_unsupported'))
        continue
    try:
        translated = translate_text(object_name, language_code)
        rows.append((language_name, translated, 'translated'))
    except Exception as exc:
        rows.append((language_name, '--', f'error: {exc}'))

import pandas as pd
pd.DataFrame(rows, columns=['Language', 'Translation', 'Status'])


## Next step

Move the reusable code into `src/`, then run `python app.py` locally for the Gradio application. The repository contains the production-oriented version of this notebook.
